**Assignment 3: Weather & To-Do List Supervisor**

**The Use Case:** A simple daily assistant that connects a Weather Agent (to check the forecast) and a To-Do List Agent (to write down tasks).

**Why one agent wouldn't be enough?**

If you tell a single agent, "Check the weather in London, and if it's raining, add 'Pack an umbrella' to my to-do list," it has to juggle two very different tasks. It might accidentally try to search the weather inside the to-do list, or try to add a task to the weather API. By using a supervisor, we keep the tools safely separated. The Weather Agent only reads data, and the To-Do Agent only writes data.

# **Setup and Tools**

In [1]:
!pip install "langchain[openai]" langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 19.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.17
    Uninstalling langchain-core-1.2.17:
      Successfully uninstalled langchain-core-1.2.17


In [6]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain.messages import HumanMessage

# Setup Model
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)


In [7]:
def create_agent(model, tools, system_prompt):
    return create_react_agent(model, tools, prompt=system_prompt)

In [31]:
import random

# This is the "Storage" for the assignment
my_todo_list = []

@tool
def get_weather(city: str) -> str:
    """Check the current weather for any city."""
    # List of realistic weather descriptions
    options = [
        f"The weather in {city} is currently 32°C and sunny.",
        f"The weather in {city} is 18°C with light rain.",
        f"The weather in {city} is 25°C and partly cloudy.",
        f"The weather in {city} is 10°C and very windy."
    ]
    # This picks one randomly so your AI has to actually 'think' before adding a task!
    return random.choice(options)

@tool
def add_task(task: str) -> str:
    """Actually saves a task into your notebook's list."""
    my_todo_list.append(task)
    return f"Success! '{task}' is now saved. You have {len(my_todo_list)} tasks total."

# **Sub-Agents and Supervisor**

In [32]:
# Create the Sub-Agents
WEATHER_PROMPT = "You are a weather assistant. Use the get_weather tool to look up forecasts."
weather_agent = create_agent(model_nemotron3_nano, [get_weather], WEATHER_PROMPT)

TODO_PROMPT = "You are a productivity assistant. Use the add_task tool to save items."
todo_agent = create_agent(model_nemotron3_nano, [add_task], TODO_PROMPT)

/tmp/ipykernel_786/2935838654.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(model, tools, prompt=system_prompt)


In [33]:
# Wrap them as tools
@tool
def check_weather_tool(request: str) -> str:
    """Use this to answer questions about the weather or forecasts."""
    result = weather_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].content

@tool
def manage_todo_tool(request: str) -> str:
    """Use this to add items or tasks to the to-do list."""
    result = todo_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].content

In [34]:
# Create the Supervisor
SUPERVISOR_PROMPT = (
    "You are a helpful logic supervisor. "
    "If a user asks about the weather, use the weather tool. "
    "If they want to save a task, use the to-do tool. "
    "If they ask for both, use both tools in the correct order."
)
supervisor_agent = create_agent(
    model_nemotron3_nano,
    tools=[check_weather_tool, manage_todo_tool],
    system_prompt=SUPERVISOR_PROMPT
)

/tmp/ipykernel_786/2935838654.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(model, tools, prompt=system_prompt)


**Test It**

In [35]:
# The Final Test
query = "What is the weather like in riyadh? If it's bad, please add 'Pack an umbrella' to my list."

result = supervisor_agent.invoke({"messages": [HumanMessage(query)]})

for message in result.get("messages", []):
    message.pretty_print()

================================ Human Message =================================

What is the weather like in riyadh? If it's bad, please add 'Pack an umbrella' to my list.
================================== Ai Message ==================================
Tool Calls:
  check_weather_tool (call_b08c94b8ca4a46c0b5ffdd1c)
 Call ID: call_b08c94b8ca4a46c0b5ffdd1c
  Args:
    request: weather in Riyadh
================================= Tool Message =================================
Name: check_weather_tool

The current weather in Riyadh is **25°C** and **partly cloudy**. It sounds like a pleasant day to be outdoors! Let me know if you'd like more details. 😊
================================== Ai Message ==================================

The weather in Riyadh is currently **25 °C** and **partly cloudy**—a pleasant condition, so you probably won’t need an umbrella today. Let me know if there’s anything else you’d like to add to your to‑do list!


In [36]:
print("Current To-Do List:", my_todo_list)

Current To-Do List: []
